In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"D:\development\aiml\tasks\part_31_supervised_ml_4\bank-full.csv", sep=";")

In [23]:
df.shape #(45211, 17)
df.isnull().sum() # no null values

np.int64(0)

In [75]:
import logging
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline  
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

config = {
    "max_depth": 5,
    "random_state": 42,
    "cv_folds": 15,
    "max_depth_values": [3, 5, 8, 12, 15, 20]
}

logging.basicConfig(
    level = logging.INFO,
    format = "[+%(relativeCreated)dms] %(levelname)s: %(message)s"
)

logger = logging.getLogger(__name__)
log = logger.info

class BankMarketingClassifier:
    def __init__(self, config: dict):
        self.max_depth = config.get("max_depth", 5)
        self.random_state = config.get("random_state", 42)
        self.cv_folds = config.get("cv_folds", 10)
        self.max_depth_values = config.get("max_depth_values", [3, 5, 8, 12, 15, 20])


    def load_data(self) -> tuple:

        df = pd.read_csv(r"D:\development\aiml\tasks\part_31_supervised_ml_4\bank-full.csv", sep=";")
        
        log(f"Data shape: {df.shape}")
        log(f"Class Distribution:\n{df["y"].value_counts()}") 
        log(f"Class Distribtuion in percent{df["y"].value_counts(normalize=True) * 100}")
        numerical_features = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
        categorical_features = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]
        log(f"Numerical Features are {numerical_features}")
        log(f"Categorical Featuresa are{categorical_features}")
        log(f"Contains no null values")
        
        X = df.drop(columns=["y"])
        y = df["y"]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=self.random_state, stratify=y
        )

        log(f"Training Set size: {X_train.shape}, Testing Set size: {X_test.shape}")
        return X_train, X_test, y_train, y_test, numerical_features, categorical_features

    def preprocess(self, X_train, X_test, numerical_features, categorical_features) -> tuple:
        encoding_preprocessor = ColumnTransformer(
            transformers=[
                ("num_encode", "passthrough", numerical_features),
                ("cat_encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
            ]
        )

        X_train_trans = encoding_preprocessor.fit_transform(X_train)
        X_test_trans = encoding_preprocessor.transform(X_test)

        log(f"Training set preprocessed: {X_train_trans.shape}, Testing set processed: {X_test_trans.shape}")

        return X_train_trans, X_test_trans, encoding_preprocessor

    def cross_validator(self, X_train, y_train, max_depth_values):
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=self.random_state)
        results = {}

        for depth in max_depth_values:
            train_acc = []
            test_acc = []

            for fold_idx (train_idx, val_idx) in enumerate (skf.split(X_train, y_train)):
                X_train = train_idx, y_train = val_idx
                clf = DecisionTreeClassifier(max_depth=depth, random_state=self.random_state)
                dtc = clf.fit(X_train, y_train)
                pred = dtc.predict(dtc)

In [76]:
df

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,unknown,yes
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,unknown,yes
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success,yes
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,unknown,no


TypeError: ColumnTransformer.__init__() got an unexpected keyword argument 'transformer'. Did you mean 'transformers'?